# Normalisasi

In [29]:
def spatial_normalize(
    origin_input_data,
    norm_div,
    norm_point=None,
    split=None,
    used_part=None,
):
    """
    Adopted from Anchor-Based Normalization (Roh et al., 2024).

    Hand is normalized relative to the palm.
    Upper limb is normalized relative to the shoulder center and width.

    Parameters
    ----------
    origin_input_data : Tensor
        Skeleton sequence with shape (T, K, C).
    norm_div : float
        Normalization divisor.
    norm_point : list, optional
        Reference keypoints for normalization.
    split : list, optional
        Cumulative keypoint split positions.
    used_part : list, optional
        Selected skeleton parts to normalize.

    Returns
    -------
    Tensor
        Spatially normalized sequence with shape (T, K, C).
    """

    # Convert input data to tensor if not already a tensor
    if not isinstance(origin_input_data, torch.Tensor):
        origin_input_data = torch.as_tensor(origin_input_data)

    # Clone tensor to avoid modifying the original tensor
    out = origin_input_data.clone().float()

    # ----------------------------------------
    # Coordinate normalization
    # ----------------------------------------

    # If split or used_part is None, return the result
    if split is None or used_part is None:
        return out

    # Add epsilon value to prevent division by zero
    eps = 1e-8
    # Add split_points[idx] + 1 to split_points to calculate the end of each part
    split_points = [0] + list(split)

    # Iterate through each part
    for idx, part in enumerate(used_part):

        # Get the start and end indices of each part
        start = split_points[idx]
        end = split_points[idx + 1]

        # Get the data for each part
        part_data = out[:, start:end, 0:2]

        # =====================================
        # LEFT HAND
        # =====================================
        if part == "left_hand":

            # Get the palm keypoint (index 0)
            palm = part_data[:, 0, :]

            # Subtract each keypoint with the palm keypoint
            out[:, start:end, 0:2] = (
                part_data
                - palm[:, None, :]
            )

        # =====================================
        # RIGHT HAND
        # =====================================
        elif part == "right_hand":

            # Get the palm keypoint (index 0)
            palm = part_data[:, 0, :]

            # Subtract each keypoint with the palm keypoint
            out[:, start:end, 0:2] = (
                part_data
                - palm[:, None, :]
            )

        # =====================================
        # UPPER LIMB
        # =====================================
        elif part == "upper_limb":

            # Define the left and right shoulder indices
            LEFT_SHOULDER = 0
            RIGHT_SHOULDER = 1

            # Get the left and right shoulder keypoints
            left_shoulder = part_data[:, LEFT_SHOULDER, :]
            right_shoulder = part_data[:, RIGHT_SHOULDER, :]

            # Get the center of the left and right shoulder keypoints
            center = (
                left_shoulder
                + right_shoulder
            ) / 2.0

            # Calculate the distance between the left and right shoulder keypoints
            shoulder_width = torch.linalg.norm(
                right_shoulder - left_shoulder,
                dim=1,
                keepdim=True,
            ).clamp_min(eps)

            # Subtract each keypoint with the center keypoint
            # and divide by the shoulder width
            out[:, start:end, 0:2] = (
                part_data
                - center[:, None, :]
            ) / shoulder_width[:, None, :]

    return out

def missing_keypoint_reconstruction(origin_input_data):
    """
    Missing keypoint reconstruction using temporal interpolation.

    Parameters
    ----------
    origin_input_data : Tensor
        Skeleton sequence with shape (T, K, C).

    Returns
    -------
    Tensor
        Reconstructed skeleton sequence.
    """
    # Create a copy of the input data for reconstruction
    result = origin_input_data.clone()

    # Extract x and y coordinates
    kp_xy = result[:, :, 0:2].cpu().numpy().astype(np.float32)
    # T = number of frames, K = number of keypoints
    T, K, _ = kp_xy.shape

    # Iterate through each keypoint
    for k in range(K):

        # Get the coordinates of the current keypoint
        coords = kp_xy[:, k, :]  # example: (T, 2)

        # Create a boolean array to mark valid frames (not missing)
        valid_mask = ~(
            (coords[:, 0] == 0) &
            (coords[:, 1] == 0)
        )
        # example: [True, True, False, True, False, ...]

        # Get the indices of valid frames
        valid_idx = np.where(valid_mask)[0] # example: [0, 1, 3, ...]

        # Check if there are valid frames for this keypoint
        if len(valid_idx) == 0:
            continue

        # Iterate through all frames for this keypoint
        for t in range(T):

            # Check if the current frame is valid
            if valid_mask[t]:
                continue
            
            # prev_arr = indices of valid frames smaller than t
            prev_arr = valid_idx[valid_idx < t] # example: [0, 1] untuk t=2
            # next_arr = indices of valid frames larger than t
            next_arr = valid_idx[valid_idx > t] # example: [3] untuk t=2

            # Check if there are valid frames before and after t
            if len(prev_arr) and len(next_arr):

                # Perform linear interpolation between the last valid frame before t and the first valid frame after t
                p = prev_arr[-1]
                n = next_arr[0]

                alpha = t - p      # distance to the previous valid frame
                beta = n - t       # distance to the next valid frame

                # Perform linear interpolation between the last valid frame before t and the first valid frame after t
                coords[t] = (
                    beta * coords[p] +
                    alpha * coords[n]
                ) / (alpha + beta)

            # If there are only valid frames before t, use the coordinates from the previous valid frame
            elif len(prev_arr):
                coords[t] = coords[prev_arr[-1]]

            # If there are only valid frames after t, use the coordinates from the next valid frame
            elif len(next_arr):
                coords[t] = coords[next_arr[0]]

        # Insert the reconstructed coordinates back into the result array
        kp_xy[:, k, :] = coords

    # Insert the reconstructed result
    result[:, :, 0:2] = torch.from_numpy(kp_xy).to(
        device=result.device,
        dtype=result.dtype
    )

    # Return the reconstructed result
    return result

def temporal_normalize(origin_input_data, target_length):
    """
    Temporal normalization by resampling to a target sequence length.

    Parameters
    ----------
    origin_input_data : Tensor
        Skeleton sequence with shape (T, K, C).
    target_length : int
        Target number of frames.

    Returns
    -------
    Tensor
        Temporally normalized sequence.
    """
    # Get the number of frames, keypoints, and channels
    T, K, C = origin_input_data.shape

    # If the number of frames is 0, raise an error
    if T == 0:
        raise ValueError("Empty sequence.")
    
    # If the number of frames is 1, repeat the sequence to the target length
    if T == 1:
        return origin_input_data.repeat(
            target_length,
            1,
            1
        )
    # If the number of frames is already equal to the target length
    if T == target_length:
        return origin_input_data.clone()

    # Convert the input data to numpy array
    data = origin_input_data.cpu().numpy()

    # Create an array of original indices
    orig_idx = np.linspace(0, T - 1, T)
    # Create an array of new indices
    new_idx = np.linspace(0, T - 1, target_length)

    # Create an array to store the result
    result = np.zeros(
        (target_length, K, C),
        dtype=data.dtype
    )

    # Interpolate each keypoint and channel
    for k in range(K):
        for c in range(C):

            # Create an interpolation function
            fn = interp1d(
                orig_idx,
                data[:, k, c],
                kind='linear'
            )

            # Interpolate the data
            result[:, k, c] = fn(new_idx)

    # Convert the result back to tensor
    return torch.from_numpy(result).to(
        device=origin_input_data.device,
        dtype=origin_input_data.dtype
    )


Augmentasi

In [30]:
class Jitter(object):
    """
    Apply Gaussian jitter (noise) to skeleton sequences.
    
    Parameters
    ----------
    std_dev : float
        Standard deviation of the Gaussian noise.
    """
    # Initialize the transform
    def __init__(self, std_dev=0.01) -> None:
        self.std_dev = std_dev
    # Apply the transform
    def __call__(self, skeleton):
        # Create Gaussian noise
        noise = np.random.normal(loc=0, scale=self.std_dev, size=skeleton.shape)
        # Add the Gaussian noise to the skeleton
        return skeleton + noise


class TemporalDropout(object):
    """
    Apply temporal dropout by randomly removing a contiguous segment of frames.
    
    Parameters
    ----------
    max_dp : float
        Maximum dropout proportion. Actual dropout length
        is between [0, vid_len * max_dp].
    """
    # Initialize the transform
    def __init__(self, max_dp=0.2):
        self.max_dp = max_dp
    # Apply the transform
    def __call__(self, clip):
        # Get the length of the clip
        vid_len = len(clip)
        # Calculate the dropout length
        dp_len = int(vid_len * self.max_dp * np.random.random())
        # Calculate the start and end indices of the dropout
        start = np.random.randint(0, vid_len - dp_len + 1)
        end = start + dp_len
        # Create a list of indices for the remaining frames
        index = list(range(0, start)) + list(range(end, vid_len))
        # Return the clip with the dropout applied
        return clip[index]

class Scale(object):
    """
    Scale skeleton sequences by applying random scaling factors.
    
    Parameters
    ----------
    scale_range : tuple
        Range of scaling factors (min, max).
    """
    # Initialize the transform
    def __init__(self, scale_range=(0.8, 1.2)) -> None:
        self.scale_range = scale_range
    # Apply the transform
    def __call__(self, skeleton):
        # Get the length of the skeleton
        T = skeleton.shape[0]
        # Generate random scale factor
        scales = np.random.uniform(*self.scale_range, size=T)
        # Scale the skeleton
        scaled_skeleton = skeleton * scales[:, np.newaxis, np.newaxis]
        # Return the scaled skeleton
        return scaled_skeleton


class TemporalRescale(object):
    """
    Temporally rescale video by resampling frames.
    
    Parameters
    ----------
    temp_scaling : float
        Temporal scaling factor. Video length is scaled 
        between [1 - temp_scaling, 1 + temp_scaling].
    """
    # Initialize the transform
    def __init__(self, temp_scaling=0.2) -> None:
        # Set the minimum and maximum lengths
        self.min_len = 32
        self.max_len = 230
        # Calculate the lower and upper bounds for temporal scaling
        self.L = 1.0 - temp_scaling
        self.U = 1.0 + temp_scaling
    # Apply the transform
    def __call__(self, clip):
        # Get the length of the clip
        vid_len = len(clip)
        # Calculate the new length
        new_len = int(vid_len * np.random.uniform(self.L, self.U))
        # Check if the new length is less than the minimum length
        if new_len < self.min_len:
            # Set the new length to the minimum length
            new_len = self.min_len
        # Check if the new length is greater than the maximum length
        if new_len > self.max_len:
            # Set the new length to the maximum length
            new_len = self.max_len
        # Check if the new length minus 4 is not divisible by 4
        if (new_len - 4) % 4 != 0:
            # Add 4 to the new length
            new_len += 4 - (new_len - 4) % 4
        if new_len <= vid_len:
            # Get the indices for the new length
            index = sorted(random.sample(range(vid_len), new_len))
        else:
            # Get the indices for the new length
            index = sorted(random.choices(range(vid_len), k=new_len))
        # Return the clip with the new length
        return clip[index]

In [31]:
import pandas as pd
import numpy as np
import torch
import random
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

def downsample(video, ratio=0.5):
    if ratio != 0.5:
        raise ValueError('This implementation only supports ratio=0.5')
    T = video.shape[0]
    if T <= 1:
        return video
    start_idx = 0 if random.uniform(0, 1) > 0.5 else 1
    idx = np.arange(start_idx, T, 2)
    return video[idx]

# Load sequence from Excel
excel_path = 'skeleton_demo.xlsx'
df = pd.read_excel(excel_path, sheet_name='P3_S01_R4')

# Convert dataframe to shape (T, K, C)
frames = sorted(df['frame'].unique())
keypoints = sorted(df['keypoint_id'].unique())
T = len(frames)
K = len(keypoints)

seq_data = np.zeros((T, K, 2))
for i, f in enumerate(frames):
    frame_data = df[df['frame'] == f]
    for j, k in enumerate(keypoints):
        kp_data = frame_data[frame_data['keypoint_id'] == k]
        if not kp_data.empty:
            seq_data[i, j, 0] = kp_data['x'].values[0]
            seq_data[i, j, 1] = kp_data['y'].values[0]

origin_input_data = torch.tensor(seq_data, dtype=torch.float32)
origin_input_data = downsample(origin_input_data, ratio=0.5)
print(f"Original input shape: {origin_input_data.shape}")


Original input shape: torch.Size([68, 54, 2])


In [32]:
def run_scenario(data, sn=False, mkr=False, tn=False, aug=None):
    out = data.clone()
    
    # Urutan logis: Missing Keypoint Recovery -> Spatial Normalization -> Temporal Normalization
    if mkr:
        out = missing_keypoint_reconstruction(out)
    if sn:
        # Menggunakan split dan used_part sesuai asumsi 86 keypoint (Holistic)
        # Karena tensor input berukuran (T, 54, 2), maka indeksnya adalah [0:21] LH, [21:42] RH, [42:54] UL
        out = spatial_normalize(
            out, 
            norm_div=1.0, 
            split=[21, 42, 54], 
            used_part=["left_hand", "right_hand", "upper_limb"]
        )
    if tn:
        out = temporal_normalize(out, target_length=64)
        
    # Augmentasi
    if aug:
        out_np = out.numpy()
        if aug == 'SJ':
            out_np = Jitter(std_dev=0.01)(out_np)
        elif aug == 'SS':
            out_np = Scale(scale_range=(0.8, 1.2))(out_np)
        elif aug == 'TD':
            out_np = TemporalDropout(max_dp=0.2)(out_np)
        elif aug == 'TR':
            out_np = TemporalRescale(temp_scaling=0.2)(out_np)
        out = torch.tensor(out_np, dtype=torch.float32)
        
    return out

# List kombinasi normalisasi (7 kombinasi)
norm_combs = [
    {'sn': True, 'mkr': False, 'tn': False},
    {'sn': False, 'mkr': True, 'tn': False},
    {'sn': False, 'mkr': False, 'tn': True},
    {'sn': True, 'mkr': True, 'tn': False},
    {'sn': True, 'mkr': False, 'tn': True},
    {'sn': False, 'mkr': True, 'tn': True},
    {'sn': True, 'mkr': True, 'tn': True}
]

scenarios = []

# M1 - M7: Tanpa Augmentasi
for nc in norm_combs:
    scenarios.append({**nc, 'aug': None})

# M8 - M11: Hanya Augmentasi
for aug in ['SJ', 'SS', 'TD', 'TR']:
    scenarios.append({'sn': False, 'mkr': False, 'tn': False, 'aug': aug})

# M12 - M39: Kombinasi Normalisasi + Augmentasi
for nc in norm_combs:
    for aug in ['SJ', 'SS', 'TD', 'TR']:
        scenarios.append({**nc, 'aug': aug})

print(f"Total skenario: {len(scenarios)}")


Total skenario: 39


In [33]:
results = []
original_T = origin_input_data.shape[0]
for i, scenario in enumerate(scenarios):
    out = run_scenario(
        origin_input_data, 
        sn=scenario['sn'], 
        mkr=scenario['mkr'], 
        tn=scenario['tn'], 
        aug=scenario['aug']
    )
    
    # --- Analysis Metrics ---
    shape = list(out.shape)
    T_out, K, C = shape
    
    frame_change_pct = ((T_out - original_T) / original_T) * 100
    
    x_data = out[:, :, 0]
    y_data = out[:, :, 1]
    
    x_range = (x_data.max() - x_data.min()).item()
    y_range = (y_data.max() - y_data.min()).item()
    
    mean_x = x_data.mean().item()
    mean_y = y_data.mean().item()
    
    var_val = out.var().item() if T_out > 1 else 0
    
    # Motion Energy (Average Frame-to-Frame displacement)
    if T_out > 1:
        diff = out[1:] - out[:-1]
        motion_energy = torch.linalg.norm(diff, dim=2).mean().item()
    else:
        motion_energy = 0.0
        
    zero_pct = ((out == 0).sum().item() / out.numel()) * 100
    
    results.append({
        'ID': f"M{i+1}",
        'SN': '\u2713' if scenario['sn'] else '',
        'MKR': '\u2713' if scenario['mkr'] else '',
        'TN': '\u2713' if scenario['tn'] else '',
        'SJ': '\u2713' if scenario['aug'] == 'SJ' else '',
        'SS': '\u2713' if scenario['aug'] == 'SS' else '',
        'TD': '\u2713' if scenario['aug'] == 'TD' else '',
        'TR': '\u2713' if scenario['aug'] == 'TR' else '',
        'Shape': str(shape),
        'Frm Chg (%)': '0' if abs(frame_change_pct) < 0.05 else f"{frame_change_pct:+.1f}",
        'Zero (%)': f"{zero_pct:.2f}",
        'Motion Eng': f"{motion_energy:.4f}",
        'X Range': f"{x_range:.2f}",
        'Y Range': f"{y_range:.2f}",
        'Var': f"{var_val:.4f}"
    })

results_df = pd.DataFrame(results)
display(results_df)

# Simpan hasil ke dalam file Excel
results_df.to_excel('order_analysis_results.xlsx', index=False)
print('Hasil analisis berhasil disimpan ke order_analysis_results.xlsx')


,ID,SN,MKR,TN,SJ,SS,TD,TR,Shape,Frm Chg (%),Zero (%),Motion Eng,X Range,Y Range,Var
0,M1,✓,,,,,,,"[68, 54, 2]",0,43.46,0.0419,1.90,4.31,0.4653
1,M2,,✓,,,,,,"[68, 54, 2]",0,0.00,0.0320,0.40,0.90,0.0602
2,M3,,,✓,,,,,"[64, 54, 2]",-5.9,37.67,0.0766,0.64,1.08,0.0950
3,M4,✓,✓,,,,,,"[68, 54, 2]",0,3.70,0.0376,1.90,4.31,0.4642
4,M5,✓,,✓,,,,,"[64, 54, 2]",-5.9,39.58,0.0411,1.89,4.30,0.4640
5,M6,,✓,✓,,,,,"[64, 54, 2]",-5.9,0.00,0.0335,0.40,0.90,0.0598
6,M7,✓,✓,✓,,,,,"[64, 54, 2]",-5.9,3.70,0.0385,1.89,4.30,0.4630
7,M8,,,,✓,,,,"[68, 54, 2]",0,0.00,0.1036,0.69,1.14,0.1021
8,M9,,,,,✓,,,"[68, 54, 2]",0,41.75,0.1384,0.72,1.27,0.1027
9,M10,,,,,,✓,,"[58, 54, 2]",-14.7,43.58,0.1070,0.64,1.08,0.1021


Hasil analisis berhasil disimpan ke order_analysis_results.xlsx
